# 💰 IMF World Economic Outlook — ELT Pipeline with dbt + DuckDB + BigQuery
**Source:** IMF Data API (free, no API key — imf.org/external/datamapper/api)
**Stack:** IMF API → S3 (Raw) → DuckDB (local transform) → dbt → BigQuery
**Pattern:** ELT (Extract-Load-Transform) — raw data lands first, dbt transforms in BigQuery
**Orchestration:** Apache Airflow (daily)

## Overview
Modern ELT pipeline using DuckDB as a local analytical engine for fast in-memory
transformation before loading to BigQuery. IMF World Economic Outlook dataset covers
GDP projections, inflation forecasts, current account balances, and fiscal positions
for 190+ countries — updated biannually (April and October).

**Why DuckDB?** Processes 190-country × 20-year dataset entirely in-memory
on a laptop-grade machine in < 2 seconds. No cluster needed.

**API:** https://imf.org/external/datamapper/api/v1 — free, no key, JSON format

## Section 1 — Imports & Configuration

In [ ]:
# pip install requests duckdb pandas google-cloud-bigquery google-cloud-storage boto3 sqlalchemy

import requests
import json
import duckdb
import logging
import pandas as pd
import boto3
from datetime import datetime, date
from pathlib import Path
from google.cloud import bigquery, storage

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('IMFPipeline')
print('✅ All imports loaded')
print(f'   DuckDB version: {duckdb.__version__}')

In [ ]:
# IMF DataMapper API — completely free, no registration
IMF_BASE_URL    = 'https://www.imf.org/external/datamapper/api/v1'

# IMF WEO indicator codes
IMF_INDICATORS = {
    'NGDP_RPCH':   'gdp_real_growth_pct',         # Real GDP growth
    'PCPIPCH':     'inflation_avg_pct',             # Inflation, avg consumer prices
    'LUR':         'unemployment_rate_pct',         # Unemployment rate
    'BCA_NGDPDP':  'current_account_pct_gdp',       # Current account balance % GDP
    'GGXCNL_NGDP': 'net_lending_pct_gdp',           # General govt net lending/borrowing
    'NGDPDPC':     'gdp_per_capita_usd',             # GDP per capita, current prices USD
    'GGXWDG_NGDP': 'gross_debt_pct_gdp',             # General govt gross debt % GDP
    'TX_RPCH':     'export_volume_change_pct',       # Export volume of goods & services
}

YEARS = list(range(2000, datetime.now().year + 2))  # 2000 to next year (includes forecasts)

# Storage
S3_BUCKET         = 'your-imf-raw-bucket'
S3_PREFIX         = 'imf/raw'
LOCAL_DUCKDB_PATH = '/tmp/imf_pipeline.duckdb'
GCP_PROJECT       = 'your-gcp-project'
BQ_DATASET        = 'imf_economic_analytics'
GCS_STAGING_BUCKET = 'your-gcs-staging-bucket'

print(f'✅ Config ready — {len(IMF_INDICATORS)} indicators, {len(YEARS)} years ({YEARS[0]}–{YEARS[-1]})')

## Section 2 — IMF API Extraction

In [ ]:
def fetch_imf_indicator(indicator_code: str) -> pd.DataFrame:
    """
    Fetch one IMF WEO indicator for ALL countries, ALL available years.
    IMF API returns JSON with nested country → year → value structure.
    """
    url = f'{IMF_BASE_URL}/{indicator_code}'
    resp = requests.get(url, timeout=30, headers={'Accept': 'application/json'})
    resp.raise_for_status()
    data = resp.json()

    values = data.get('values', {}).get(indicator_code, {})
    records = []
    for country_code, year_values in values.items():
        for year_str, value in year_values.items():
            if value is None:
                continue
            try:
                records.append({
                    'country_code':    country_code.upper(),
                    'indicator_code':  indicator_code,
                    'indicator_name':  IMF_INDICATORS.get(indicator_code, indicator_code),
                    'year':            int(year_str),
                    'value':           float(value),
                    'is_forecast':     int(year_str) > datetime.now().year - 1,
                    'batch_date':      str(date.today()),
                    'ingested_at':     datetime.utcnow().isoformat(),
                    'source':          'imf-datamapper-api'
                })
            except (ValueError, TypeError):
                continue

    df = pd.DataFrame(records)
    logger.info(f'  ✅ {indicator_code}: {len(df)} records')
    return df

def fetch_country_metadata() -> pd.DataFrame:
    """
    Fetch country labels and region info from IMF DataMapper.
    """
    url = f'{IMF_BASE_URL}/countries'
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    countries = resp.json().get('countries', {})
    records = [
        {'country_code': code.upper(), 'country_name': info.get('label', code)}
        for code, info in countries.items()
    ]
    df = pd.DataFrame(records)
    logger.info(f'✅ Country metadata: {len(df)} countries')
    return df

def extract_all_to_s3(execution_date: str) -> dict:
    """
    ELT Step 1: Extract raw data → upload to S3 as Parquet (raw zone).
    """
    s3 = boto3.client('s3')
    counts = {}

    # Country metadata
    meta_df = fetch_country_metadata()
    s3_key = f'{S3_PREFIX}/countries/dt={execution_date}/countries.parquet'
    s3.put_object(Bucket=S3_BUCKET, Key=s3_key, Body=meta_df.to_parquet(index=False))
    counts['countries'] = len(meta_df)

    # All indicators
    all_records = []
    for code, name in IMF_INDICATORS.items():
        df = fetch_imf_indicator(code)
        all_records.append(df)

    combined = pd.concat(all_records, ignore_index=True)
    s3_key = f'{S3_PREFIX}/indicators/dt={execution_date}/weo_indicators.parquet'
    s3.put_object(Bucket=S3_BUCKET, Key=s3_key, Body=combined.to_parquet(index=False))
    counts['indicator_records'] = len(combined)

    logger.info(f'✅ ELT Extract complete: {counts}')
    return counts

# extract_all_to_s3(str(date.today()))

## Section 3 — DuckDB Local Transform (ELT Transform Layer)

In [ ]:
"""
DuckDB acts as a local, in-memory analytical engine.
It reads Parquet from S3 directly using the httpfs extension,
runs SQL transforms, and writes cleaned data ready for BigQuery.
No cluster, no server — DuckDB runs entirely in-process.
"""

def transform_with_duckdb(execution_date: str) -> pd.DataFrame:
    """
    ELT Step 2: DuckDB reads S3 Parquet → pivot → enrich → return as DataFrame.
    """
    con = duckdb.connect(LOCAL_DUCKDB_PATH)

    # Load httpfs extension to read S3 directly
    con.execute("INSTALL httpfs; LOAD httpfs;")
    con.execute("SET s3_region='ap-southeast-1';")

    # Load raw Parquet from S3
    con.execute(f"""
        CREATE OR REPLACE VIEW raw_indicators AS
        SELECT * FROM read_parquet(
            's3://{S3_BUCKET}/{S3_PREFIX}/indicators/dt={execution_date}/*.parquet'
        )
    """)
    con.execute(f"""
        CREATE OR REPLACE VIEW raw_countries AS
        SELECT * FROM read_parquet(
            's3://{S3_BUCKET}/{S3_PREFIX}/countries/dt={execution_date}/*.parquet'
        )
    """)

    # Pivot indicators → wide format
    pivot_sql = """
        CREATE OR REPLACE TABLE stg_weo_wide AS
        PIVOT raw_indicators
        ON indicator_name
        USING FIRST(value)
        GROUP BY country_code, year, is_forecast, batch_date
    """
    con.execute(pivot_sql)

    # Enrich: join country names, classify regions, compute health score
    enrich_sql = """
        CREATE OR REPLACE TABLE mart_weo_enriched AS
        SELECT
            w.country_code,
            c.country_name,
            w.year,
            w.is_forecast,
            ROUND(w.gdp_real_growth_pct, 2)         AS gdp_growth_pct,
            ROUND(w.inflation_avg_pct, 2)            AS inflation_pct,
            ROUND(w.unemployment_rate_pct, 2)        AS unemployment_pct,
            ROUND(w.current_account_pct_gdp, 2)      AS current_account_pct,
            ROUND(w.gross_debt_pct_gdp, 2)           AS gross_debt_pct,
            ROUND(w.gdp_per_capita_usd, 0)           AS gdp_per_capita_usd,
            ROUND(w.export_volume_change_pct, 2)     AS export_growth_pct,
            -- Economic health composite score
            ROUND(
                COALESCE(w.gdp_real_growth_pct, 0)     * 0.35
              - COALESCE(w.inflation_avg_pct, 0)       * 0.25
              - COALESCE(w.unemployment_rate_pct, 0)   * 0.25
              + COALESCE(w.current_account_pct_gdp, 0) * 0.15
            , 2)                                     AS health_score,
            -- Income classification
            CASE
                WHEN w.gdp_per_capita_usd >= 13000 THEN 'High income'
                WHEN w.gdp_per_capita_usd >= 4500  THEN 'Upper-middle'
                WHEN w.gdp_per_capita_usd >= 1200  THEN 'Lower-middle'
                ELSE 'Low income'
            END                                      AS income_group,
            w.batch_date,
            CURRENT_TIMESTAMP                        AS loaded_at
        FROM stg_weo_wide w
        LEFT JOIN raw_countries c ON w.country_code = c.country_code
        WHERE w.year BETWEEN 2000 AND EXTRACT(YEAR FROM CURRENT_DATE) + 1
    """
    con.execute(enrich_sql)

    result_df = con.execute('SELECT * FROM mart_weo_enriched').df()
    con.close()

    logger.info(f'✅ DuckDB transform complete: {len(result_df)} rows')
    return result_df

# result = transform_with_duckdb(str(date.today()))
# print(result.head())

## Section 4 — Load to BigQuery (ELT Load Step)

In [ ]:
def load_to_bigquery(df: pd.DataFrame, execution_date: str) -> int:
    """
    ELT Step 3: Load DuckDB-transformed DataFrame to BigQuery.
    Uses WRITE_TRUNCATE for idempotent daily refresh.
    """
    bq = bigquery.Client(project=GCP_PROJECT)
    table_id = f'{GCP_PROJECT}.{BQ_DATASET}.weo_enriched'

    # Delete today's batch for idempotency
    bq.query(f"""
        DELETE FROM `{table_id}`
        WHERE batch_date = '{execution_date}'
    """).result()

    job = bq.load_table_from_dataframe(
        df, table_id,
        job_config=bigquery.LoadJobConfig(
            write_disposition=bigquery.WriteDisposition.WRITE_APPEND,
            schema=[
                bigquery.SchemaField('country_code',      'STRING'),
                bigquery.SchemaField('country_name',      'STRING'),
                bigquery.SchemaField('year',              'INTEGER'),
                bigquery.SchemaField('is_forecast',       'BOOLEAN'),
                bigquery.SchemaField('gdp_growth_pct',    'FLOAT64'),
                bigquery.SchemaField('inflation_pct',     'FLOAT64'),
                bigquery.SchemaField('unemployment_pct',  'FLOAT64'),
                bigquery.SchemaField('health_score',      'FLOAT64'),
                bigquery.SchemaField('income_group',      'STRING'),
                bigquery.SchemaField('gdp_per_capita_usd','FLOAT64'),
                bigquery.SchemaField('batch_date',        'DATE'),
                bigquery.SchemaField('loaded_at',         'TIMESTAMP'),
            ]
        )
    )
    job.result()
    logger.info(f'✅ Loaded {len(df)} rows → BigQuery {table_id}')
    return len(df)

# ELT Full Run:
# counts  = extract_all_to_s3(str(date.today()))
# df      = transform_with_duckdb(str(date.today()))
# loaded  = load_to_bigquery(df, str(date.today()))

## Section 5 — Pipeline Summary

| Layer | Technology | Role |
|---|---|---|
| **Extract** | IMF DataMapper API | 8 WEO indicators × 190+ countries × 25 years |
| **Raw Zone** | Amazon S3 (Parquet) | Raw landing zone — immutable, partitioned by date |
| **Transform** | **DuckDB** | Local in-memory SQL engine — pivot, enrich, score |
| **Load** | Google BigQuery | Append with idempotent DELETE+INSERT |
| **Reporting** | BigQuery Analytics | dbt-ready marts for dashboards |
| **Orchestration** | Airflow | Daily 04:00 WIB |

**ELT vs ETL:**
- ETL: transform before loading (needs cluster)
- ELT: load raw first, transform in warehouse using SQL (DuckDB → BigQuery)

**DuckDB advantages:**
- Reads S3 Parquet natively via httpfs extension
- PIVOT syntax built-in (no manual CASE WHEN needed)
- Processes 50,000+ rows in < 1 second in-memory
- Zero infrastructure cost — runs on any machine

**Volume:** ~50,000 records/run (8 indicators × 190 countries × 25 years)